# Gold Layer Daily Sales Detail

Personal project notebook to build `hive_metastore.praxas_gold.dailysalesdetail`.

Discovered silver source roles used in this notebook:

* `hive_metastore.praxas_silver.f_salesline` is the main sales line fact and the driving grain.
* `hive_metastore.praxas_silver.salesorder` adds order-header dates, customer references, ship-to references, and order-level status/type context.
* `hive_metastore.praxas_silver.customer` adds customer attributes such as name, geography, type, terms, and account manager.
* `hive_metastore.praxas_silver.warehouse` adds warehouse business attributes.
* `hive_metastore.praxas_silver.custshiptoaddr` is optional enrichment for ship-to geography and address context.

Design notes:

* Gold grain is daily aggregated sales detail, not raw line grain.
* Sales date is derived using practical fallback logic that prioritizes realized business dates, then booked/planned dates, then audit timestamps.
* The notebook includes defaults, lightweight DQ checks, final dedup, and Delta merge logic.
* This notebook is authored only and is intentionally not executed here.

In [0]:
# Databricks notebook source
# =============================================================================
# GOLD LAYER : Daily Sales Detail Aggregation (UNITY CATALOG PROD STANDARD)
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, TimestampType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import logging

# ──────────────────────────────────────────────────────────────────────────────
# 1. ENVIRONMENT CONFIGURATION & OPTIMIZATIONS
# ──────────────────────────────────────────────────────────────────────────────
# Force Delta to handle file sizing automatically!
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoOptimize.autoCompact.enabled", "true")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("gold_dailysalesdetail")
print("🟡 INITIALIZING PROD GOLD PIPELINE : DAILY SALES DETAIL FACT")

CATALOG_NAME = "prx"
SILVER_SCHEMA = "prx_silver"
GOLD_SCHEMA = "prx_gold"

STORAGE_ACCOUNT = "stpraxaslakehouse"

# Fully Qualified Unity Catalog Silver Sources
SILVER_SALESLINE_TABLE  = f"{CATALOG_NAME}.{SILVER_SCHEMA}.f_salesline"
SILVER_SALESORDER_TABLE = f"{CATALOG_NAME}.{SILVER_SCHEMA}.salesorder"
SILVER_CUSTOMER_TABLE   = f"{CATALOG_NAME}.{SILVER_SCHEMA}.customer"
SILVER_WAREHOUSE_TABLE  = f"{CATALOG_NAME}.{SILVER_SCHEMA}.warehouse"
SILVER_SHIPTO_TABLE     = f"{CATALOG_NAME}.{SILVER_SCHEMA}.custshiptoaddr"

# Gold Target
GOLD_TABLE = f"{CATALOG_NAME}.{GOLD_SCHEMA}.dailysalesdetail"
GOLD_PATH  = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/delta/praxas/dailysalesdetail"
MERGE_KEYS = ["DailySalesDetailKey"]

# Ensure Gold Schema Exists Securely
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG_NAME}.{GOLD_SCHEMA}")

# Capture true executing user for Auditing
executing_user = spark.sql("SELECT current_user()").collect()[0][0]

# ──────────────────────────────────────────────────────────────────────────────
# 2. UTILITIES
# ──────────────────────────────────────────────────────────────────────────────
def safe_str(c):
    return F.coalesce(F.trim(F.col(c).cast(StringType())), F.lit(""))

def safe_float(c):
    return F.coalesce(F.col(c).cast(DoubleType()), F.lit(0.0))

def first_not_blank_str(*cols):
    exprs = [F.when(F.col(c).isNotNull() & (F.trim(F.col(c).cast(StringType())) != ""), F.trim(F.col(c).cast(StringType()))) for c in cols]
    return F.coalesce(*(exprs + [F.lit("")]))

def first_not_blank_ts(*cols):
    exprs = [F.when(F.col(c).isNotNull() & (F.trim(F.col(c).cast(StringType())) != ""), F.col(c).cast(TimestampType())) for c in cols]
    return F.coalesce(*exprs)

def deterministic_hash(*cols):
    return F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast(StringType()), F.lit("")) for c in cols]), 256)

def build_merge_condition(keys):
    return " AND ".join([f"tgt.{k} = src.{k}" for k in keys])

def merge_to_delta(df, table, path, keys):
    cond = build_merge_condition(keys)
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
    
    if spark.catalog.tableExists(table):
        delta_tbl = DeltaTable.forName(spark, table)
        (delta_tbl.alias("tgt")
            .merge(df.alias("src"), cond)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
    else:
        df.write.format("delta").mode("overwrite").option("path", path).saveAsTable(table)

# ──────────────────────────────────────────────────────────────────────────────
# 3. READ SILVER SOURCES (UNITY CATALOG)
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Reading Silver source tables...")

salesline_df  = spark.table(SILVER_SALESLINE_TABLE).dropDuplicates(["SOSrcId", "SOLnSrcId"]).alias("l")
salesorder_df = spark.table(SILVER_SALESORDER_TABLE).dropDuplicates(["SOSrcID"]).alias("o")
customer_df   = spark.table(SILVER_CUSTOMER_TABLE).dropDuplicates(["CustSrcId"]).alias("c")
warehouse_df  = spark.table(SILVER_WAREHOUSE_TABLE).dropDuplicates(["WHSrcId"]).alias("w")
shipto_df     = spark.table(SILVER_SHIPTO_TABLE).dropDuplicates(["CustShipToAddrSrcId"]).alias("s")

# ──────────────────────────────────────────────────────────────────────────────
# 4. BUSINESS LOGIC & BASE FACT JOIN
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Joining silver sources and deriving daily sales detail base rows...")

joined_df = (salesline_df
    .join(salesorder_df, F.col("l.SOSrcId") == F.col("o.SOSrcID"), "left")
    .join(F.broadcast(customer_df), F.col("o.CustSrcID") == F.col("c.CustSrcId"), "left")
    .join(F.broadcast(warehouse_df), F.col("l.WHSrcId") == F.col("w.WHSrcId"), "left")
    .join(F.broadcast(shipto_df), F.col("o.CustShipToAddrSrcID") == F.col("s.CustShipToAddrSrcId"), "left")
)

sales_dt_ts_expr = first_not_blank_ts(
    "l.ActlShipDt", "o.CompletedDt", "o.SODt", "l.PlanShipDt", 
    "l.PromiseDt", "o.ShipDt", "o.ReqShipDt", "l.CancDt", 
    "o.CancDt", "l.CreatedDt", "o.UpdatedDt"
)

base_df = joined_df.filter(F.upper(safe_str("l.IsDeletedFl")) != "YES").select(
    safe_str("l.SOSrcId").alias("SOSrcId"),
    safe_str("l.SOLnSrcId").alias("SOLnSrcId"),
    safe_str("l.SOKey").alias("SOKey"),
    safe_str("l.SOLnKey").alias("SOLnKey"),
    sales_dt_ts_expr.alias("SalesDtTs"),
    F.to_date(sales_dt_ts_expr).alias("SalesDt"),
    safe_str("o.SOTrxnType").alias("SOTrxnType"),
    safe_str("o.SOTrxnTypeDesc").alias("SOTrxnTypeDesc"),
    safe_str("o.SOType").alias("SOType"),
    safe_str("o.SOTypeDesc").alias("SOTypeDesc"),
    safe_str("o.SOOrdCat").alias("SOOrdCat"),
    safe_str("o.SOOrdCatDesc").alias("SOOrdCatDesc"),
    safe_str("o.SOStatus").alias("SOStatus"),
    safe_str("o.SOProcessStatus").alias("SOProcessStatus"),
    safe_str("l.SOLnStatus").alias("SOLnStatus"),
    safe_str("l.SOLnShipStatus").alias("SOLnShipStatus"),
    safe_str("l.InvLnStatus").alias("InvLnStatus"),
    safe_str("o.SalesRepName").alias("SalesRepName"),
    safe_str("o.CustSrcID").alias("CustSrcId"),
    safe_str("c.CustKey").alias("CustKey"),
    safe_str("c.CustName").alias("CustName"),
    safe_str("c.CustType").alias("CustType"),
    safe_str("c.CustTypeDesc").alias("CustTypeDesc"),
    safe_str("c.CustStatus").alias("CustStatus"),
    safe_str("c.TermsCd").alias("TermsCd"),
    safe_str("c.AcctMngr").alias("AcctMngr"),
    safe_str("c.City").alias("CustCity"),
    safe_str("c.State").alias("CustState"),
    safe_str("c.Country").alias("CustCountry"),
    safe_str("c.ZipCd").alias("CustZipCd"),
    safe_str("o.CustShipToAddrSrcID").alias("CustShipToAddrSrcId"),
    first_not_blank_str("s.City", "c.City").alias("ShipToCity"),
    first_not_blank_str("s.State", "c.State").alias("ShipToState"),
    first_not_blank_str("s.Country", "c.Country").alias("ShipToCountry"),
    first_not_blank_str("s.ZipCd", "c.ZipCd").alias("ShipToZipCd"),
    safe_str("l.WHSrcId").alias("WHSrcId"),
    safe_str("w.WHKey").alias("WHKey"),
    safe_str("w.WHName").alias("WHName"),
    safe_str("w.WHStatus").alias("WHStatus"),
    safe_str("w.DivSrcId").alias("DivSrcId"),
    safe_str("w.DivKey").alias("DivKey"),
    safe_str("l.ItemSrcId").alias("ItemSrcId"),
    safe_str("l.ItemKey").alias("ItemKey"),
    safe_str("l.ItemDesc").alias("ItemDesc"),
    safe_str("l.ItemType").alias("ItemType"),
    safe_str("l.UOM").alias("UOM"),
    safe_str("l.UOMDesc").alias("UOMDesc"),
    safe_str("l.SOLnCat").alias("SOLnCat"),
    safe_str("l.SOLnCatDesc").alias("SOLnCatDesc"),
    safe_str("l.SOLnCostType").alias("SOLnCostType"),
    safe_str("l.SOLnCostTypeDesc").alias("SOLnCostTypeDesc"),
    safe_str("l.CustCurr").alias("CustCurr"),
    safe_str("l.LocCurr").alias("LocCurr"),
    safe_float("l.SOLnOrgQty").alias("SOLnOrgQty"),
    safe_float("l.SOLnOrdQty").alias("SOLnOrdQty"),
    safe_float("l.SOLnCancQty").alias("SOLnCancQty"),
    safe_float("l.SOLnRvsdQty").alias("SOLnRvsdQty"),
    safe_float("l.ShippedQty").alias("ShippedQty"),
    safe_float("l.InvLnQty").alias("InvLnQty"),
    safe_float("l.DlvryQty").alias("DlvryQty"),
    safe_float("l.BacklogQty").alias("BacklogQty"),
    safe_float("l.SOLnValCC").alias("NetSalesValCC"),
    safe_float("l.SOLnValLC").alias("NetSalesValLC"),
    safe_float("l.SOLnValUS").alias("NetSalesValUS"),
    safe_float("l.SOLnDiscValCC").alias("DiscountValCC"),
    safe_float("l.SOLnDiscValLC").alias("DiscountValLC"),
    safe_float("l.SOLnDiscValUS").alias("DiscountValUS"),
    safe_float("l.SOLnCostCC").alias("CostValCC"),
    safe_float("l.SOLnCostLC").alias("CostValLC"),
    safe_float("l.SOLnCostUS").alias("CostValUS"),
    safe_float("l.SOLnTaxValCC").alias("TaxValCC"),
    safe_float("l.SOLnTaxValLC").alias("TaxValLC"),
    safe_float("l.SOLnTaxValUS").alias("TaxValUS"),
    F.coalesce(F.greatest(F.col("l.UpdatedDt"), F.col("o.UpdatedDt"), F.col("c.UpdatedDt"), F.col("w.UpdatedDt"), F.col("s.UpdatedDt")), F.current_timestamp()).alias("SourceUpdatedDt")
)

# ──────────────────────────────────────────────────────────────────────────────
# 5. GOLD AGGREGATION
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Aggregating to daily sales detail grain...")

grain_cols = [
    "SalesDt", "SOTrxnType", "SOTrxnTypeDesc", "SOType", "SOTypeDesc", "SOOrdCat", 
    "SOOrdCatDesc", "SOStatus", "SOProcessStatus", "SOLnStatus", "SOLnShipStatus", 
    "InvLnStatus", "SalesRepName", "CustSrcId", "CustKey", "CustName", "CustType", 
    "CustTypeDesc", "CustStatus", "TermsCd", "AcctMngr", "CustCity", "CustState", 
    "CustCountry", "CustZipCd", "CustShipToAddrSrcId", "ShipToCity", "ShipToState", 
    "ShipToCountry", "ShipToZipCd", "WHSrcId", "WHKey", "WHName", "WHStatus", 
    "DivSrcId", "DivKey", "ItemSrcId", "ItemKey", "ItemDesc", "ItemType", "UOM", 
    "UOMDesc", "SOLnCat", "SOLnCatDesc", "SOLnCostType", "SOLnCostTypeDesc", 
    "CustCurr", "LocCurr"
]

gold_df = base_df.groupBy(*grain_cols).agg(
    F.count("SOLnSrcId").alias("SalesLineCnt"),
    F.countDistinct("SOSrcId").alias("SalesOrderCnt"),
    F.sum("SOLnOrgQty").alias("OrigQty"),
    F.sum("SOLnOrdQty").alias("OrderQty"),
    F.sum("SOLnCancQty").alias("CancelledQty"),
    F.sum("SOLnRvsdQty").alias("RevisedQty"),
    F.sum("ShippedQty").alias("ShippedQty"),
    F.sum("InvLnQty").alias("InvoiceQty"),
    F.sum("DlvryQty").alias("DeliveryQty"),
    F.sum("BacklogQty").alias("BacklogQty"),
    F.sum("NetSalesValCC").alias("NetSalesValCC"),
    F.sum("NetSalesValLC").alias("NetSalesValLC"),
    F.sum("NetSalesValUS").alias("NetSalesValUS"),
    F.sum("DiscountValCC").alias("DiscountValCC"),
    F.sum("DiscountValLC").alias("DiscountValLC"),
    F.sum("DiscountValUS").alias("DiscountValUS"),
    F.sum("CostValCC").alias("CostValCC"),
    F.sum("CostValLC").alias("CostValLC"),
    F.sum("CostValUS").alias("CostValUS"),
    F.sum("TaxValCC").alias("TaxValCC"),
    F.sum("TaxValLC").alias("TaxValLC"),
    F.sum("TaxValUS").alias("TaxValUS"),
    F.sum(F.when(F.col("SOLnStatus") == "Open", F.lit(1)).otherwise(F.lit(0))).alias("OpenLineCnt"),
    F.sum(F.when(F.col("SOLnStatus") == "Completed", F.lit(1)).otherwise(F.lit(0))).alias("CompletedLineCnt"),
    F.sum(F.when(F.col("SOLnStatus") == "Cancelled", F.lit(1)).otherwise(F.lit(0))).alias("CancelledLineCnt"),
    F.max("SourceUpdatedDt").alias("MaxSourceUpdatedDt")
).withColumn("GrossMarginCC", F.col("NetSalesValCC") - F.col("CostValCC")) \
 .withColumn("GrossMarginLC", F.col("NetSalesValLC") - F.col("CostValLC")) \
 .withColumn("GrossMarginUS", F.col("NetSalesValUS") - F.col("CostValUS")) \
 .withColumn("SalesYear", F.year(F.col("SalesDt"))) \
 .withColumn("SalesMonth", F.month(F.col("SalesDt"))) \
 .withColumn("SalesDay", F.dayofmonth(F.col("SalesDt"))) \
 .withColumn("SalesWeekOfYear", F.weekofyear(F.col("SalesDt"))) \
 .withColumn("SalesMonthName", F.date_format(F.col("SalesDt"), "MMMM")) \
 .withColumn("DailySalesDetailKey", deterministic_hash(*grain_cols)) \
 .withColumn("SysUpdatedDt", F.current_timestamp()) \
 .withColumn("SysUpdatedBy", F.lit(executing_user))

# ──────────────────────────────────────────────────────────────────────────────
# 6. DATA QUALITY & FINAL DEDUP
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Applying gold-level DQ and final dedup...")

dq_df = gold_df.withColumn(
    "DQ_Reason", F.array_remove(F.array(
        F.when(F.col("DailySalesDetailKey") == "", F.lit("MISSING_HASH_KEY")),
        F.when(F.col("SalesDt").isNull(), F.lit("MISSING_SALES_DT"))
    ), None)
).withColumn("DQ_Status", F.when(F.size(F.col("DQ_Reason")) > 0, F.lit(2)).otherwise(F.lit(0)))

# 🔥 DAG PROTECTION
dq_df.cache()

valid_df = dq_df.filter(F.col("DQ_Status") == 0).drop("DQ_Reason", "DQ_Status")
error_df = dq_df.filter(F.col("DQ_Status") == 2)

logger.info(f"✅ Valid gold rows ready for merge: {valid_df.count():,}")
logger.info(f"❌ Gold DQ exceptions: {error_df.count():,}")

window_spec = Window.partitionBy(*MERGE_KEYS).orderBy(F.col("MaxSourceUpdatedDt").desc_nulls_last())

final_df = valid_df.withColumn("rn", F.row_number().over(window_spec)).filter(F.col("rn") == 1).drop("rn")

# ──────────────────────────────────────────────────────────────────────────────
# 7. MERGE TO GOLD DELTA
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Merging daily sales detail to Gold Delta table...")

merge_to_delta(final_df, GOLD_TABLE, GOLD_PATH, MERGE_KEYS)

dq_df.unpersist()

print("🎉 PROD GOLD PIPELINE COMPLETED SUCCESSFULLY!")

# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT * FROM prx.prx_gold.dailysalesdetail ORDER BY SalesDt DESC LIMIT 10;